# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
#loading the data from last week
!pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"

df_content = con.sql(f"SELECT * FROM read_parquet('{base}/dim_content.parquet')").df()
df_clients = con.sql(f"SELECT * FROM read_parquet('{base}/dim_clients.parquet')").df()
df_march = con.sql(f"SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
agg_df = df_march.groupby(['client_hash_id', 'content_hash_id']).agg(
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    gsc_sum_position=('gsc_sum_position', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
).reset_index()

agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']

print(f"agg_df shape: {agg_df.shape}")
agg_df.head()

agg_df shape: (331437, 8)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_sessions,ga4_engaged_sessions,gsc_avg_position
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,0,0,0,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,0,0,0,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,0,0,0,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9,0,0,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,0,0,0,NaN


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.